In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Dense , Embedding , Lambda
from sklearn.metrics.pairwise import cosine_similarity
import random
import re

In [2]:
# Data Collection :
df = pd.read_csv("/Users/aayushagarwal/Downloads/LP IV Updated Datasets/CBOW/twitter_dataset.csv" , encoding="latin-1")
tweets = df["Text"].astype(str).tolist()
print(len(tweets))
print(tweets[:3])

10000
['Party least receive say or single. Prevent prevent husband affect. May himself cup style evening protect. Effect another themselves stage perform.\nPossible try tax share style television with. Successful much sell development economy effect.', 'Hotel still Congress may member staff. Media draw buy fly. Identify on another turn minute would.\nLocal subject way believe which question some message. Own all imagine join agency indicate.', 'Nice be her debate industry that year. Film where generation push discover partner level.\nNearly money store style may enjoy. Kid discuss blue save. Model another about along.\nEverybody especially dinner character yard.']


In [3]:
# Tokenization :
tokenizer = Tokenizer(num_words=3000)   # keep top 3000 words
tokenizer.fit_on_texts(tweets)
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = 3000
sequences = tokenizer.texts_to_sequences(tweets)
print("Vocab Size : " , vocab_size)

Vocab Size :  3000


In [4]:
# Generate CBOW Training Data :
window_size = 2
contexts , targets = [] , []

for seq in sequences:
    for i in range(window_size , len(seq) - window_size):
        contexts.append(seq[i - window_size : i] + seq[i + 1 : i + window_size + 1])
        targets.append(seq[i])

contexts = np.array(contexts)
targets = np.array(targets)
print("Training samples : " , len(contexts))

Training samples :  299853


In [5]:
# CBOW Model :
model = keras.Sequential([
    Embedding(vocab_size , 100),
    Lambda(lambda x : tf.reduce_mean(x , axis=1)),
    Dense(vocab_size , activation="softmax")
])

model.compile(optimizer="adam" , loss="sparse_categorical_crossentropy" , metrics=["accuracy"])
model.fit(contexts , targets , epochs=5 , batch_size=256)

Epoch 1/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 9.4046e-04 - loss: 7.0061
Epoch 2/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0011 - loss: 6.8853
Epoch 3/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0012 - loss: 6.8772
Epoch 4/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0020 - loss: 6.8629
Epoch 5/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0031 - loss: 6.8368


In [6]:
# Word Embeddings : (weights of the Embedding layer)
embeddings = model.layers[0].get_weights()[0]
print(embeddings.shape)

(3000, 100)


In [7]:
# Document Vectors : average of the embeddings of all words in the document
def doc_vector(seq):
    seq = [w for w in seq if w < vocab_size]
    if len(seq) == 0:
        return np.zeros(embeddings.shape[1])
    return embeddings[seq].mean(axis=0)

doc_vectors = np.array([doc_vector(seq) for seq in sequences])
print(doc_vectors.shape)

(10000, 100)


In [9]:
# Recommendation : cosine similarity between user input and every document
def recommend(text , top_n=5):
    seq = tokenizer.texts_to_sequences([(text)])[0]
    query = doc_vector(seq).reshape(1 , -1)
    sims = cosine_similarity(query , doc_vectors)[0]
    top = sims.argsort()[::-1][:top_n]
    for i in top:
        print(round(sims[i] , 3) , " | " , tweets[i][:120])

recommend("I love this new phone")

0.775  |  North carry population message rich interesting sit. Benefit people indicate likely our business and. Place laugh of rad
0.772  |  Firm sit hour foot significant. Glass indeed wonder behavior large who subject. Radio rich energy officer most phone.
Av
0.765  |  Claim carry husband else watch. Recent three figure. Improve cost bag must particularly crime different.
Build reduce dr
0.762  |  Fund site week. Tough program else mouth huge.
Black might cost to fire. If may purpose detail. View early work case com
0.762  |  Entire anything else enjoy best. Develop inside knowledge technology lawyer less phone. Capital important determine toge


In [10]:
# Prediction : recommend tweets similar to a random tweet
n = random.randint(0 , len(tweets) - 1)
print("Input : " , tweets[n][:120])
print("Recommended :")
recommend(tweets[n] , top_n=6)

Input :  Policy successful our upon. Might billion civil line.
Fact rock short music sell morning live. Late significant which ra
Recommended :
1.0  |  Policy successful our upon. Might billion civil line.
Fact rock short music sell morning live. Late significant which ra
0.963  |  True consider culture play. Baby lay he fall send writer no economy.
Agree care expect market nice. Rest read explain ai
0.961  |  Region begin my effect range.
Skin American those about Mrs affect pull.
News benefit send go quality may. Which signifi
0.957  |  Party evidence expert he evening and. Finish safe body certainly.
Collection own what boy form executive. Particular soc
0.957  |  Always when show record whether. Rule little hair process new because capital. Law reflect writer before.
Stand never ha
0.957  |  Result least act way carry. Home reveal response economy religious guy body.
Air finish under century. Set agent maybe.

